# Fine-tune EfficientNet-B0 on Egyptian Modalink frames (Colab)

**Run this only after:**
1. Frame extract finished → `MyDrive/MasterData/egypt_modalink_frames/` + `manifest.csv`
2. Kaggle EfficientNet-B0 finished → you have **`best_model_efficientnet_b0.h5`** (full model, not weights-only)

**Important (avoids layer mismatch):**
- Upload the Kaggle file as-is
- This notebook uses **`load_model(...)` only** — it does **not** rebuild EfficientNet and call `load_weights`
- If you see `EfficientNetB0_finetune` + “132 vs 11 layers”, you are on the **wrong notebook/cell**

**What this does**
- Loads your EfficientNet-B0 baseline
- Fine-tunes on Egyptian frames (Video Emotion labels)
- Person-level train/val/test
- Saves `best_model_efficientnet_egypt_ft.h5`


In [ ]:
# 1) Mount Drive (same Google account that has MasterData)
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

from pathlib import Path
print("MyDrive top folders:")
for p in sorted(Path("/content/drive/MyDrive").iterdir())[:40]:
    print(" -", p.name)

In [ ]:
# 2) CONFIG — edit only if your paths differ
from pathlib import Path

MYDRIVE = Path("/content/drive/MyDrive")

# Frames from extract notebook (MasterData default; Masterdata also accepted)
CANDIDATES = [
    MYDRIVE / "MasterData" / "egypt_modalink_frames",
    MYDRIVE / "Masterdata" / "egypt_modalink_frames",
    MYDRIVE / "Master Data" / "egypt_modalink_frames",
]
FRAMES_ROOT = next((p for p in CANDIDATES if p.exists()), CANDIDATES[0])

# Baseline EfficientNet from Kaggle — upload to Colab or put on Drive
BASELINE_H5 = Path("/content/best_model_efficientnet_b0.h5")
# Or from Drive, e.g.:
# BASELINE_H5 = MYDRIVE / "MasterData" / "models" / "best_model_efficientnet_b0.h5"

OUT_DIR = MYDRIVE / "MasterData" / "models"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Must match Kaggle EfficientNet training
IMAGE_SIZE = 96
BATCH_SIZE = 32
SEED = 42
TRAIN_RATIO, VAL_RATIO = 0.70, 0.15  # rest = test

STAGE1_EPOCHS = 12   # head only
STAGE2_EPOCHS = 20   # unfreeze last EfficientNet blocks
UNFREEZE_LAST_N = 40

print("FRAMES_ROOT exists:", FRAMES_ROOT.exists(), "→", FRAMES_ROOT)
print("BASELINE_H5 exists:", BASELINE_H5.exists(), "→", BASELINE_H5)
print("OUT_DIR:", OUT_DIR)
assert FRAMES_ROOT.exists(), f"Frames folder not found. Checked: {CANDIDATES}"
assert (FRAMES_ROOT / "manifest.csv").exists(), f"Missing manifest.csv under {FRAMES_ROOT}"
assert BASELINE_H5.exists(), (
    "Upload best_model_efficientnet_b0.h5 to /content/ or set BASELINE_H5 to your Drive path"
)

In [ ]:
# 3) Install / imports
!pip -q install scikit-learn pandas opencv-python-headless

import json
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import seaborn as sns

tf.keras.utils.set_random_seed(SEED)
print("TF:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
# 4) Load manifest + person-level split
manifest = pd.read_csv(FRAMES_ROOT / "manifest.csv")
print("manifest rows:", len(manifest))
print("columns:", list(manifest.columns))
print(manifest.head(3))

def pick_col(df, options):
    for c in options:
        if c in df.columns:
            return c
    raise KeyError(f"None of {options} in columns={list(df.columns)}")

EMOTION_COL = pick_col(manifest, ["emotion", "label", "class"])
PATH_COL = pick_col(manifest, ["image_path", "rel_path", "frame_path", "path", "filepath"])
PERSON_COL = pick_col(manifest, ["person_id", "person_key", "person"])

df = manifest.copy()
df["emotion"] = df[EMOTION_COL].astype(str).str.strip().str.lower()
df = df[~df["emotion"].isin(["nan", "none", "", "ambiguous", "unknown"])].copy()

CANON = ["angry", "disgust", "fear", "happy", "neutral", "sad", "surprise"]
ALIAS = {
    "anger": "angry", "angry": "angry",
    "disgust": "disgust",
    "fear": "fear",
    "happiness": "happy", "happy": "happy",
    "neutral": "neutral",
    "sadness": "sad", "sad": "sad",
    "surprise": "surprise",
}
df["emotion"] = df["emotion"].map(lambda x: ALIAS.get(x, x))
df = df[df["emotion"].isin(CANON)].copy()

def resolve_path(p):
    p = Path(str(p))
    if p.is_absolute() and p.exists():
        return p
    cand = FRAMES_ROOT / p
    if cand.exists():
        return cand
    cand2 = MYDRIVE / p
    if cand2.exists():
        return cand2
    return cand

df["abs_path"] = df[PATH_COL].map(resolve_path)
df = df[df["abs_path"].map(lambda p: Path(p).exists())].copy()
print("usable frames:", len(df))
print("class counts:\n", df["emotion"].value_counts())
print("unique persons:", df[PERSON_COL].nunique())

meta_path = BASELINE_H5.with_name("metrics.json")
if not meta_path.exists():
    meta_path = BASELINE_H5.parent / "metrics.json"
if meta_path.exists():
    meta = json.loads(meta_path.read_text())
    CLASS_NAMES = list(meta.get("class_names", CANON))
    print("CLASS_NAMES from metrics.json:", CLASS_NAMES)
else:
    CLASS_NAMES = CANON
    print("CLASS_NAMES (default FER order):", CLASS_NAMES)

df = df[df["emotion"].isin(CLASS_NAMES)].copy()
label_to_idx = {c: i for i, c in enumerate(CLASS_NAMES)}
df["y"] = df["emotion"].map(label_to_idx)

persons = df[PERSON_COL].unique()
train_p, temp_p = train_test_split(persons, train_size=TRAIN_RATIO, random_state=SEED)
val_frac = VAL_RATIO / (1 - TRAIN_RATIO)
val_p, test_p = train_test_split(temp_p, train_size=val_frac, random_state=SEED)

split_map = {**{p: "train" for p in train_p}, **{p: "val" for p in val_p}, **{p: "test" for p in test_p}}
df["split"] = df[PERSON_COL].map(split_map)

for s in ["train", "val", "test"]:
    sub = df[df["split"] == s]
    print(f"{s}: frames={len(sub)} persons={sub[PERSON_COL].nunique()} classes={sub['emotion'].value_counts().to_dict()}")

assert set(df.loc[df.split=="train", PERSON_COL]).isdisjoint(df.loc[df.split=="test", PERSON_COL])
print("Person leakage check: OK")

In [ ]:
# 5) tf.data pipelines (EfficientNet preprocess)
AUTOTUNE = tf.data.AUTOTUNE

def load_image(path, y):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMAGE_SIZE, IMAGE_SIZE])
    img = tf.cast(img, tf.float32)
    img = tf.keras.applications.efficientnet.preprocess_input(img)
    return img, y

def make_ds(split, shuffle=False, augment=False):
    sub = df[df["split"] == split]
    paths = sub["abs_path"].astype(str).tolist()
    ys = sub["y"].astype(np.int32).tolist()
    ds = tf.data.Dataset.from_tensor_slices((paths, ys))
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    if augment:
        def aug(x, y):
            x = tf.image.random_flip_left_right(x)
            x = tf.image.random_brightness(x, 0.1)
            x = tf.image.random_contrast(x, 0.9, 1.1)
            return x, y
        ds = ds.map(aug, num_parallel_calls=AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(min(len(paths), 2048), seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_ds("train", shuffle=True, augment=True)
val_ds = make_ds("val")
test_ds = make_ds("test")

y_train = df.loc[df["split"] == "train", "y"].values
cw = compute_class_weight("balanced", classes=np.arange(len(CLASS_NAMES)), y=y_train)
CLASS_WEIGHT = {int(i): float(w) for i, w in enumerate(cw)}
print("Class weights:", CLASS_WEIGHT)

In [ ]:
# 6) Load FULL baseline model (do NOT rebuild + load_weights)
# Wrong pattern that causes "132 vs 11 layers":
#   model = build_new_efficientnet(); model.load_weights(h5)
# Correct:
#   model = tf.keras.models.load_model(h5)

print("Loading:", BASELINE_H5)
try:
    model = tf.keras.models.load_model(str(BASELINE_H5), compile=False)
except Exception as e:
    raise RuntimeError(
        "Could not load full model from .h5. "
        "Re-download best_model_efficientnet_b0.h5 from Kaggle Output "
        "(the full model save, not a weights-only file). "
        f"Original error: {e}"
    ) from e

def has_efficientnet(m):
    if "efficientnet" in m.name.lower():
        return True
    for l in m.layers:
        if "efficientnet" in l.name.lower():
            return True
        if hasattr(l, "layers") and any("efficientnet" in x.name.lower() for x in l.layers):
            return True
    return False

print("Loaded model name:", model.name)
print("Top-level layer count:", len(model.layers))
print("Top-level layers:", [l.name for l in model.layers])
assert has_efficientnet(model), (
    "Loaded file is NOT EfficientNet. Use best_model_efficientnet_b0.h5 from the fixed Kaggle notebook."
)
assert "finetune" not in model.name.lower() or has_efficientnet(model), "Unexpected model"
print("EfficientNet check: OK")
model.summary()

# Nested EfficientNet base (typical: ~7–12 top-level layers, base is ONE nested layer)
eff_base = None
for layer in model.layers:
    if "efficientnet" in layer.name.lower() and hasattr(layer, "layers"):
        eff_base = layer
        break
print("EfficientNet base layer:", None if eff_base is None else f"{eff_base.name} ({len(eff_base.layers)} inner layers)")
if eff_base is None:
    print("WARNING: nested EfficientNet base not found — Stage 2 will unfreeze top layers only")


In [ ]:
# 7) Stage 1 — freeze backbone, train head
if eff_base is not None:
    eff_base.trainable = False
else:
    for layer in model.layers[:-4]:
        layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

ckpt1 = str(OUT_DIR / "ckpt_egypt_stage1.keras")
cbs1 = [
    tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=4, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6),
    tf.keras.callbacks.ModelCheckpoint(ckpt1, monitor="val_accuracy", save_best_only=True),
]

print("Stage 1: head only")
hist1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=STAGE1_EPOCHS,
    class_weight=CLASS_WEIGHT,
    callbacks=cbs1,
)

In [ ]:
# 8) Stage 2 — unfreeze last N EfficientNet layers
if eff_base is not None:
    eff_base.trainable = True
    n = len(eff_base.layers)
    freeze_until = max(0, n - UNFREEZE_LAST_N)
    for i, layer in enumerate(eff_base.layers):
        layer.trainable = i >= freeze_until
    print(f"Unfroze last {UNFREEZE_LAST_N} of {n} EfficientNet layers")
else:
    for layer in model.layers:
        layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

ckpt2 = str(OUT_DIR / "best_model_efficientnet_egypt_ft.keras")
cbs2 = [
    tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-7),
    tf.keras.callbacks.ModelCheckpoint(ckpt2, monitor="val_accuracy", save_best_only=True),
]

print("Stage 2: fine-tune")
hist2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=STAGE2_EPOCHS,
    class_weight=CLASS_WEIGHT,
    callbacks=cbs2,
)

In [ ]:
# 9) Person-level TEST evaluation (this is the number that matters)
y_true, y_pred = [], []
for xb, yb in test_ds:
    probs = model.predict(xb, verbose=0)
    y_true.extend(yb.numpy().tolist())
    y_pred.extend(probs.argmax(axis=1).tolist())

y_true = np.array(y_true)
y_pred = np.array(y_pred)
acc = (y_true == y_pred).mean()
macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)

print(f"TEST accuracy: {acc:.4f}")
print(f"TEST macro-F1: {macro_f1:.4f}")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=list(range(len(CLASS_NAMES))))
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title("EfficientNet-B0 Egypt FT — person-level TEST")
plt.ylabel("true")
plt.xlabel("pred")
plt.tight_layout()
plt.show()

In [ ]:
# 10) Save final .h5 + metrics (download / keep on Drive)
assert has_efficientnet(model), "Abort save — model is not EfficientNet"

h5_path = OUT_DIR / "best_model_efficientnet_egypt_ft.h5"
model.save(str(h5_path))
print("Saved:", h5_path)

metrics = {
    "backbone": "EfficientNetB0",
    "stage": "egypt_finetune_video_emotion",
    "image_size": IMAGE_SIZE,
    "class_names": CLASS_NAMES,
    "frames_root": str(FRAMES_ROOT),
    "baseline": str(BASELINE_H5),
    "test_accuracy": float(acc),
    "test_macro_f1": float(macro_f1),
    "n_train": int((df.split == "train").sum()),
    "n_val": int((df.split == "val").sum()),
    "n_test": int((df.split == "test").sum()),
    "n_persons_train": int(df.loc[df.split == "train", PERSON_COL].nunique()),
    "n_persons_val": int(df.loc[df.split == "val", PERSON_COL].nunique()),
    "n_persons_test": int(df.loc[df.split == "test", PERSON_COL].nunique()),
}
(OUT_DIR / "egypt_ft_metrics.json").write_text(json.dumps(metrics, indent=2))
print(json.dumps(metrics, indent=2))

from google.colab import files
files.download(str(h5_path))
print("Also on Drive:", h5_path)
print("Next local smoke test:")
print("python webcam_test_h5.py --model artifacts/models/best_model_efficientnet_egypt_ft.h5")

## After it finishes — send me
1. `TEST accuracy` + `TEST macro-F1`
2. Classification report (per-class)
3. Confirm `egypt_ft_metrics.json` says `"backbone": "EfficientNetB0"`

**Ignore train accuracy for thesis claims** — only person-level TEST matters.